# Stock forecasting baseline
Portfolio edition, 31 October 2026. Original coursework is preserved outside this folder.
Run from this notebook folder after installing requirements and placing the documented inputs in data/.
This baseline/audit is separate from the original coursework scores. All performance labels refer to its own holdout.


In [1]:
"""Chronological research baseline; separate from the original coursework scores."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.metrics import f1_score,accuracy_score,mean_squared_error

ROOT=Path.cwd() if (Path.cwd()/'data/stock').exists() else Path(__file__).resolve().parent
DATA=ROOT/'data/stock'

def make_features(stock):
    x=stock.copy().sort_values(['stock_id','month_id']).reset_index(drop=True)
    numeric=[c for c in x if c not in ['stock_id','month_id']]
    x[numeric]=x[numeric].apply(pd.to_numeric,errors='coerce')
    # Reindex each stock to calendar months: a shift is one month, not one observed row.
    grids=[]
    for key,g in x.groupby('stock_id',sort=True):
        g=g.set_index(pd.PeriodIndex(g['month_id'].str.replace('_','-'),freq='M')).drop(columns='month_id')
        if not g.index.is_unique:raise ValueError('Duplicate stock/month records')
        g=g.reindex(pd.period_range(g.index.min(),g.index.max(),freq='M'))
        values=g[numeric].shift(1) # predict this month using information from previous months only
        for w in [3,6]:
            values[f'return_mean_{w}m']=g['intramonth_return'].shift(1).rolling(w,min_periods=2).mean()
            values[f'return_std_{w}m']=g['intramonth_return'].shift(1).rolling(w,min_periods=2).std()
        values['stock_id']=key;values['month_id']=values.index.strftime('%Y_%m');grids.append(values.reset_index(drop=True))
    return pd.concat(grids,ignore_index=True)

stock=pd.read_csv(DATA/'stock_data.csv');targets=pd.read_csv(DATA/'training_targets.csv')
features=make_features(stock)
# Future perturbation test: later raw values must not change earlier features.
cut=sorted(stock['month_id'].unique())[-6];changed=stock.copy();cols=[c for c in stock if c not in ['stock_id','month_id']]
changed.loc[changed.month_id>cut,cols]=999999
probe=make_features(changed)
pd.testing.assert_frame_equal(features[features.month_id<=cut].reset_index(drop=True),probe[probe.month_id<=cut].reset_index(drop=True))
joined=features.merge(targets,on=['stock_id','month_id'],how='inner',validate='one_to_one').dropna(subset=['outperform_binary','excess_return'])
cols=[c for c in features if c not in ['stock_id','month_id']]
joined=joined.dropna(subset=cols,how='all');months=sorted(joined.month_id.unique());validation=months[-6:]
results=[]
for month in validation:
    train=joined[joined.month_id<month];test=joined[joined.month_id==month]
    clf=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),RandomForestClassifier(n_estimators=80,max_depth=7,min_samples_leaf=10,random_state=42,n_jobs=2))
    reg=make_pipeline(SimpleImputer(strategy='median',add_indicator=True),RandomForestRegressor(n_estimators=80,max_depth=7,min_samples_leaf=10,random_state=42,n_jobs=2))
    clf.fit(train[cols],train.outperform_binary.astype(int));reg.fit(train[cols],train.excess_return)
    pred=clf.predict(test[cols]);value=reg.predict(test[cols]);majority=int(train.outperform_binary.mode().iloc[0]);mean=float(train.excess_return.mean())
    results.append({'month':month,'train_rows':len(train),'validation_rows':len(test),'f1':float(f1_score(test.outperform_binary,pred,zero_division=0)),'baseline_f1':float(f1_score(test.outperform_binary,np.full(len(test),majority),zero_division=0)),'accuracy':float(accuracy_score(test.outperform_binary,pred)),'rmse':float(np.sqrt(mean_squared_error(test.excess_return,value))),'baseline_rmse':float(np.sqrt(mean_squared_error(test.excess_return,np.full(len(test),mean))))})
out={'kind':'new chronological portfolio baseline','features':'prior-month numeric stock features and past-only rolling returns; train-fold median imputation','validation':'six expanding-window calendar-month holdouts; fixed parameters, no tuning on holdout','future_invariance_check':True,'rows':results,'mean_f1':float(np.mean([r['f1'] for r in results])),'mean_baseline_f1':float(np.mean([r['baseline_f1'] for r in results])),'mean_rmse':float(np.mean([r['rmse'] for r in results])),'mean_baseline_rmse':float(np.mean([r['baseline_rmse'] for r in results]))}
(ROOT/'outputs').mkdir(exist_ok=True);(ROOT/'outputs/stock-results.json').write_text(json.dumps(out,indent=2));print(json.dumps(out,indent=2))


{
  "kind": "new chronological portfolio baseline",
  "features": "prior-month numeric stock features and past-only rolling returns; train-fold median imputation",
  "validation": "six expanding-window calendar-month holdouts; fixed parameters, no tuning on holdout",
  "future_invariance_check": true,
  "rows": [
    {
      "month": "2023_01",
      "train_rows": 21306,
      "validation_rows": 616,
      "f1": 0.5142857142857142,
      "baseline_f1": 0.0,
      "accuracy": 0.5308441558441559,
      "rmse": 0.1054818032581169,
      "baseline_rmse": 0.10626749239883183
    },
    {
      "month": "2023_02",
      "train_rows": 21922,
      "validation_rows": 616,
      "f1": 0.3830734966592428,
      "baseline_f1": 0.0,
      "accuracy": 0.5503246753246753,
      "rmse": 0.07170399767526962,
      "baseline_rmse": 0.07194617831003618
    },
    {
      "month": "2023_03",
      "train_rows": 22538,
      "validation_rows": 616,
      "f1": 0.20803782505910165,
      "baseline_f1": 0.0